In [3]:
!pip install openpyxl pandas

In [45]:
def limpiar_boletin(path):
    import pandas as pd

    raw = pd.read_excel(path, header=None, engine="openpyxl")

    start_row = None
    for i in range(len(raw)):
        row = raw.iloc[i].astype(str).str.lower()
        if row.str.contains("rude").any():
            start_row = i
            break

    if start_row is None:
        raise ValueError(f"No se detectó cabecera en {path}")

    df = pd.read_excel(path, skiprows=start_row, engine="openpyxl")

    df = df.dropna(axis=1, how="all")
    df = df.dropna(axis=0, how="all")

    df.columns = df.columns.astype(str).str.strip().str.lower()

    rude_col = None
    for col in df.columns:
        if "rude" in col.replace(".", ""):
            rude_col = col
            break

    if not rude_col:
        raise ValueError(f"No se encontró RUDE en {path}")

    df = df.rename(columns={rude_col: "rude"})

    materias_map = {
        "pa": "com_lenguajes",
        "pa.1": "cs_sociales",
        "pa.2": "edu_fisica",
        "pa.3": "edu_musical",
        "pa.4": "art_plasticas",
        "pa.5": "matematica",
        "pa.6": "tec_tecnologica",
        "pa.7": "cs_naturales",
        "pa.8": "valores_religion"
    }

    df = df.rename(columns=materias_map)

    df.columns = (
        df.columns
        .str.replace(" ", "_")
        .str.replace("á","a").str.replace("é","e")
        .str.replace("í","i")
        .str.replace("ó","o")
        .str.replace("ú","u")
    )

    for col in materias_map.values():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[df["rude"].notna()]

    return df


In [54]:
def limpiar_boletin_v2(path):
    import pandas as pd
    import os

    # 1. Leemos el archivo "crudo" para buscar cabeceras y metadatos
    raw = pd.read_excel(path, header=None, engine="openpyxl")

    # --- NUEVO: EXTRACCIÓN DE METADATOS ---
    # Basado en la estructura estandar del Ministerio que vimos en tu archivo:
    # Fila 0, Col 1 -> Gestión (Ej: 2022)
    # Fila 1, Col 3 -> Año Escolaridad (Ej: CUARTO)
    # Fila 1, Col 5 -> Paralelo (Ej: A)
    try:
        gestion_val = str(raw.iloc[0, 1]).strip()
        curso_val = str(raw.iloc[1, 3]).strip().upper()
        paralelo_val = str(raw.iloc[1, 5]).strip().upper()
    except Exception as e:
        print(f"Advertencia: No se pudieron extraer metadatos exactos de {path}. Se dejarán vacíos.")
        gestion_val, curso_val, paralelo_val = None, None, None
    # --------------------------------------

    # 2. Lógica original para encontrar dónde empieza la tabla
    start_row = None
    for i in range(len(raw)):
        # Convertimos a string para evitar error si hay nulos
        row = raw.iloc[i].astype(str).str.lower()
        if row.str.contains("rude").any():
            start_row = i
            break

    if start_row is None:
        raise ValueError(f"No se detectó cabecera en {path}")

    # 3. Leemos la tabla limpia desde la fila encontrada
    df = pd.read_excel(path, skiprows=start_row, engine="openpyxl")

    # Limpieza básica de filas/columnas vacías
    df = df.dropna(axis=1, how="all")
    df = df.dropna(axis=0, how="all")

    # Normalización de nombres de columnas
    df.columns = df.columns.astype(str).str.strip().str.lower()

    # Búsqueda de la columna RUDE
    rude_col = None
    for col in df.columns:
        if "rude" in col.replace(".", ""):
            rude_col = col
            break

    if not rude_col:
        raise ValueError(f"No se encontró RUDE en {path}")

    df = df.rename(columns={rude_col: "rude"})

    # Mapeo de materias
    materias_map = {
        "pa": "com_lenguajes",
        "pa.1": "cs_sociales",
        "pa.2": "edu_fisica",
        "pa.3": "edu_musical",
        "pa.4": "art_plasticas",
        "pa.5": "matematica",
        "pa.6": "tec_tecnologica",
        "pa.7": "cs_naturales",
        "pa.8": "valores_religion"
    }

    df = df.rename(columns=materias_map)

    # Limpieza de caracteres raros en los nombres de columna
    df.columns = (
        df.columns
        .str.replace(" ", "_")
        .str.replace("á","a").str.replace("é","e")
        .str.replace("í","i")
        .str.replace("ó","o")
        .str.replace("ú","u")
        .str.replace("\n", "") # Agregué esto por si hay saltos de linea
    )

    # Conversión a numérico de las notas
    for col in materias_map.values():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Filtrar filas sin RUDE
    df = df[df["rude"].notna()]

    # --- NUEVO: INYECCIÓN DE METADATOS AL DATAFRAME ---
    # Asignamos los valores extraídos al principio a nuevas columnas
    df.insert(0, 'gestion', gestion_val)
    df.insert(1, 'anio_escolaridad', curso_val)
    df.insert(2, 'paralelo', paralelo_val)
    
    # Opcional: Agregar el nombre del archivo origen para rastreo
    df['archivo_origen'] = os.path.basename(path)
    # --------------------------------------------------

    return df

In [55]:
import glob, os, pandas as pd, re

BASE_DIR = os.path.abspath("..")
DATA_DIR = os.path.join(BASE_DIR, "data/boletines_extraidos_excel")

files = glob.glob(os.path.join(DATA_DIR, "**", "*.xlsx"), recursive=True)
files = [f for f in files if not os.path.basename(f).startswith("~$")]

print("Archivos encontrados:", len(files))

dfs = []

for f in files:
    try:
        df = limpiar_boletin_v2(f)
        dfs.append(df)
    except Exception as e:
        print("⚠️ Error en:", f)
        print("   ", e)

if len(dfs) == 0:
    raise ValueError("No se pudo procesar ningún archivo")

dataset = pd.concat(dfs, ignore_index=True)
print("Total registros:", len(dataset))


Archivos encontrados: 36
Total registros: 1118


In [56]:
# // Alumnos rezagados
materias = [
    "com_lenguajes",
    "cs_sociales",
    "edu_fisica",
    "edu_musical",
    "art_plasticas",
    "matematica",
    "tec_tecnologica",
    "cs_naturales",
    "valores_religion"
]

dataset["rezago"] = (dataset[materias] < 51).any(axis=1).astype(int)

dataset["rezago"].value_counts()



rezago
0    1095
1      23
Name: count, dtype: int64

In [57]:
dataset.groupby("rezago")[materias].mean()


,com_lenguajes,cs_sociales,edu_fisica,edu_musical,art_plasticas,matematica,tec_tecnologica,cs_naturales,valores_religion
rezago,,,,,,,,,
0,74.486594,74.479643,78.019861,77.409136,75.926514,73.472691,76.271102,75.040715,70.329692
1,45.956522,49.173913,58.739130,57.608696,53.434783,47.043478,53.130435,49.869565,49.826087


In [59]:
# aniadir nueva columna

dataset["num_materias_reprobadas"] = (dataset[materias] < 51).sum(axis=1)
dataset["promedio_general"] = dataset[materias].mean(axis=1)

In [60]:
dataset

,gestion,anio_escolaridad,paralelo,nro.,paterno,materno,nombres,rude,gen,fecha_nac,...,edu_musical,art_plasticas,matematica,tec_tecnologica,cs_naturales,valores_religion,archivo_origen,rezago,num_materias_reprobadas,promedio_general
0,2022,PRIMERO,B,1,NaN,FERNANDEZ,AYELEN BELEN,809800682020086,F,09-07-2015,...,84.0,87.0,89.0,90.0,90.0,91.0,boletin_centralizador_80980027_12_Primero_B_20...,0,0,88.333333
1,2022,PRIMERO,B,2,NaN,HINOJOSA,ARELI LUCIA,609000282018010,F,31-10-2013,...,NaN,NaN,NaN,NaN,NaN,NaN,boletin_centralizador_80980027_12_Primero_B_20...,0,0,NaN
2,2022,PRIMERO,B,3,ALANES,FUENTES,DESIRE CIELO,809804752020027,F,11-08-2015,...,79.0,69.0,67.0,69.0,66.0,92.0,boletin_centralizador_80980027_12_Primero_B_20...,0,0,73.777778
3,2022,PRIMERO,B,4,ALVAREZ,SEJAS,REYCHEL,809801842020002,F,10-09-2015,...,73.0,54.0,55.0,54.0,52.0,65.0,boletin_centralizador_80980027_12_Primero_B_20...,0,0,59.333333
4,2022,PRIMERO,B,5,APAZA,CARVAJAL,KAROLAY FELISA,809801202020120,F,28-04-2016,...,56.0,52.0,51.0,51.0,51.0,41.0,boletin_centralizador_80980027_12_Primero_B_20...,1,2,51.444444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1113,2023,PRIMERO,A,28,TORREZ,COLQUE,EUGENE MATIAS,8098008320214763,M,17-03-2017,...,63.0,66.0,60.0,61.0,63.0,56.0,boletin_centralizador_80980027_12_Primero_A_20...,0,0,61.666667
1114,2023,PRIMERO,A,29,TORRICO,GONZALES,AGUSTIN MATTEO,8098055120215690,M,10-04-2017,...,87.0,74.0,71.0,73.0,78.0,84.0,boletin_centralizador_80980027_12_Primero_A_20...,0,0,77.333333
1115,2023,PRIMERO,A,30,VARGAS,MONTENEGRO,GUSTAVO ANDRE,8098060020213014,M,29-12-2016,...,87.0,79.0,76.0,71.0,71.0,76.0,boletin_centralizador_80980027_12_Primero_A_20...,0,0,76.222222
1116,2023,PRIMERO,A,31,VILLEGAS,DELGADILLO,SEBASTIAN\nNICOLAS,8098051020225260,M,30-12-2016,...,68.0,83.0,86.0,86.0,87.0,83.0,boletin_centralizador_80980027_12_Primero_A_20...,0,0,83.333333


In [61]:
# Descargar dataset final en csv y excel
dataset.to_csv(
    "../data/dataset/primaria_dataset.csv",
    index=False,
    encoding="utf-8"
)
dataset.to_excel(
    "../data/dataset/primaria_dataset.xlsx",
    index=False
)

In [62]:
dataset.shape
dataset.columns
dataset["rezago"].value_counts()


rezago
0    1095
1      23
Name: count, dtype: int64

In [63]:
print(dataset.columns.tolist())


['gestion', 'anio_escolaridad', 'paralelo', 'nro.', 'paterno', 'materno', 'nombres', 'rude', 'gen', 'fecha_nac', 'lugnac', 'numeroc.i.', 'estadomatricula', 'com_lenguajes', 'cs_sociales', 'edu_fisica', 'edu_musical', 'art_plasticas', 'matematica', 'tec_tecnologica', 'cs_naturales', 'valores_religion', 'archivo_origen', 'rezago', 'num_materias_reprobadas', 'promedio_general']
